### Round 0 Analysis - Davide
Provide an analysis of both products, focusing mainly on TOMATOES since EMERALDS are just Market Making
I'll be adding log prices and log returns to the dataframe, moving on with the modelling of returns and other hypotheses

In [ ]:
import os
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.stats as stats

from pathlib import Path
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

import prosperity4
from prosperity4.utils.dataloader import (
    load_trading_data,
    get_product_data,
    get_day_data,
    get_product_day_data,
    get_price_data,
    get_order_book_data,
    get_volume_data,
    convert_timestamp,
)

plt.style.use("dark_background")
sns.set_palette("pastel")

### Data Loading

In [ ]:
REPO_ROOT = Path(prosperity4.__file__).parents[1]
DATA_FOLDER = REPO_ROOT / "prosperity4" / "round0" / "data"
ROUND_NUM = 0
DAYS = [-2, -1]

data = load_trading_data(DATA_FOLDER, ROUND_NUM, DAYS)
prices_df = data.get("prices")
trades_df = data.get("trades")

print("Prices Shape :", prices_df.shape if prices_df is not None else None)
print("Trades Shape :", trades_df.shape if trades_df is not None else None)
print("\n--- Prices Head ---")
display(prices_df.head())
print("\n--- Trades Head ---")
display(trades_df.head())

### Splitting the datasets based on the products

In [ ]:
products = prices_df["product"].unique()

emeralds_prices_df = prices_df[prices_df["product"] == "EMERALDS"]
tomatoes_prices_df = prices_df[prices_df["product"] == "TOMATOES"]
emeralds_trades_df = trades_df[trades_df["symbol"] == "EMERALDS"]
tomatoes_trades_df = trades_df[trades_df["symbol"] == "TOMATOES"]

tomatoes_trades_df.shape

In [ ]:
'''
Here we are copying the original dataframes, removing the buyer, seller and currency from the trades, summing if there are orders at the
same timestamp and price, then renaming the price and quantity column to market order price and quantity so we know these are bots trades,
then we are merging the 2 datasets so we have all the data on prices and volume, if at that timestamp there's a bot's trade, there's also the 
data for these trades.
At the end we are converting the timestamps so we have timestamp as index
'''
emeralds_trades = emeralds_trades_df.copy()
tomatoes_trades = tomatoes_trades_df.copy()

# Grouping and merging the EMERALDS so we only have 1 dataset containing both prices and trades, sorted by timestamp
emeralds_trades = emeralds_trades.drop(columns = ["buyer", "seller", "currency"])
emeralds_trades = emeralds_trades.groupby(["timestamp", "price"], as_index = False).agg({"quantity": "sum"})
emeralds_trades = emeralds_trades.rename(columns = { 
                                        "price": "market order price",
                                        "quantity": "market order quantity"})
emeralds = emeralds_prices_df.merge(emeralds_trades[["timestamp", "market order price", "market order quantity"]],
                                on = "timestamp",
                                how = "left")

# Grouping and merging the EMERALDS so we only have 1 dataset containing both prices and trades, sorted by timestamp
tomatoes_trades = tomatoes_trades.drop(columns = ["buyer", "seller", "currency"])
tomatoes_trades = tomatoes_trades.groupby(["timestamp", "price"], as_index = False).agg({"quantity": "sum"})
tomatoes_trades = tomatoes_trades.rename(columns = { 
                                        "price": "market order price",
                                        "quantity": "market order quantity"})
tomatoes = tomatoes_prices_df.merge(tomatoes_trades[["timestamp", "market order price", "market order quantity"]],
                                on = "timestamp",
                                how = "left")

# Convert to continuous timeframe so we don't have 2 separate days
emeralds = convert_timestamp(emeralds)
tomatoes = convert_timestamp(tomatoes)

emeralds.head()


# Price and Trades plots

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(emeralds["t"][-500:], emeralds["mid_price"][-500:], color = "white", alpha = 0.5)
plt.plot(emeralds["t"][-500:], emeralds["ask_price_1"][-500:], color = "red", alpha = 1)
plt.plot(emeralds["t"][-500:], emeralds["ask_price_2"][-500:], color = "orange", alpha = 0.5)
plt.plot(emeralds["t"][-500:], emeralds["ask_price_3"][-500:], color = "salmon", alpha = 0.25)
plt.plot(emeralds["t"][-500:], emeralds["bid_price_1"][-500:], color = "lime", alpha = 1)
plt.plot(emeralds["t"][-500:], emeralds["bid_price_2"][-500:], color = "green", alpha = 0.5)
plt.plot(emeralds["t"][-500:], emeralds["bid_price_3"][-500:], color = "darkgreen", alpha = 0.25)
plt.show()

In [ ]:
# This considers the mid price
plt.figure(figsize=(15,5))
plt.plot(tomatoes["t"][-500:], tomatoes["mid_price"][-500:], color = "white", alpha = 0.5)
plt.plot(tomatoes["t"][-500:], tomatoes["ask_price_1"][-500:], color = "red", alpha = 1)
plt.plot(tomatoes["t"][-500:], tomatoes["ask_price_2"][-500:], color = "orange", alpha = 0.5)
plt.plot(tomatoes["t"][-500:], tomatoes["ask_price_3"][-500:], color = "salmon", alpha = 0.25)
plt.plot(tomatoes["t"][-500:], tomatoes["bid_price_1"][-500:], color = "lime", alpha = 1)
plt.plot(tomatoes["t"][-500:], tomatoes["bid_price_2"][-500:], color = "green", alpha = 0.5)
plt.plot(tomatoes["t"][-500:], tomatoes["bid_price_3"][-500:], color = "darkgreen", alpha = 0.25)
plt.show()

In [ ]:
bid_vol_cols   = ['bid_volume_1', 'bid_volume_2', 'bid_volume_3']
bid_price_cols = ['bid_price_1',  'bid_price_2',  'bid_price_3']
ask_vol_cols   = ['ask_volume_1', 'ask_volume_2', 'ask_volume_3']
ask_price_cols = ['ask_price_1',  'ask_price_2',  'ask_price_3']

# idxmax(axis=1) returns the column name with the highest volume per row (NaN-safe)
bid_vol_winner = tomatoes[bid_vol_cols].idxmax(axis=1)
ask_vol_winner = tomatoes[ask_vol_cols].idxmax(axis=1)

# Map volume column name → corresponding price column name
bid_map = dict(zip(bid_vol_cols, bid_price_cols))
ask_map = dict(zip(ask_vol_cols, ask_price_cols))

# Row-wise lookup: for each row, read the price at the winning level
tomatoes['max_vol_bid_price'] = [tomatoes.at[idx, bid_map[col]] for idx, col in bid_vol_winner.items()]
tomatoes['max_vol_ask_price'] = [tomatoes.at[idx, ask_map[col]] for idx, col in ask_vol_winner.items()]

tomatoes["fv"] = (tomatoes["max_vol_bid_price"] + tomatoes["max_vol_ask_price"]) / 2

In [ ]:
# This considers the fv as the midpoint between max vol bid and ask
plt.figure(figsize=(15,5))
plt.plot(tomatoes["t"][-500:], tomatoes["fv"][-500:], color = "white", alpha = 0.5)
plt.plot(tomatoes["t"][-500:], tomatoes["ask_price_1"][-500:], color = "red", alpha = 1)
plt.plot(tomatoes["t"][-500:], tomatoes["ask_price_2"][-500:], color = "orange", alpha = 0.5)
plt.plot(tomatoes["t"][-500:], tomatoes["ask_price_3"][-500:], color = "salmon", alpha = 0.25)
plt.plot(tomatoes["t"][-500:], tomatoes["bid_price_1"][-500:], color = "lime", alpha = 1)
plt.plot(tomatoes["t"][-500:], tomatoes["bid_price_2"][-500:], color = "green", alpha = 0.5)
plt.plot(tomatoes["t"][-500:], tomatoes["bid_price_3"][-500:], color = "darkgreen", alpha = 0.25)

### Volatility Arbitrage

In [ ]:
tomatoes["log_prices"] = np.log(tomatoes["mid_price"])
tomatoes["log_returns"] = tomatoes["log_prices"] - tomatoes["log_prices"].shift(1)
